In [1]:
##### Creates a layer which is the aggregate of all crops from MapSPAM
# unit = mt

import os
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors

In [3]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Set file paths
raster_folder = f"/{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_production"
save_path = f"{cd}/Data/Clean/Production/crop_production_metric_ton_2020.tif"


In [ ]:
##### Produce global map for crop production 

# Get paths to all rasters in MapSPAM
raster_paths = glob(os.path.join(raster_folder, "*.tif"))
raster_paths.sort()  

sum_array = None
profile = None

# Use first raster as template
template_path = raster_paths[0]

with rasterio.open(template_path) as template_src:
    profile = template_src.profile.copy()
    template_data = template_src.read(1).astype(np.float32)

    # Initialize sum array based on template data
    sum_array = np.zeros_like(template_data, dtype=np.float32)

    # Add template raster first
    if template_src.nodata is not None:
        template_data = np.where(template_data == template_src.nodata, np.nan, template_data)
    sum_array += np.nan_to_num(template_data)

    # Loop through remaining rasters
    for path in raster_paths[1:]:
        with rasterio.open(path) as src:

            # Prepare array to match template
            resampled = np.empty_like(template_data, dtype=np.float32)

            # Reproject/resample to match template
            reproject(
                source=src.read(1),
                destination=resampled,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=template_src.transform,
                dst_crs=template_src.crs,
                resampling=Resampling.bilinear
            )

            # Handle nodata (nan_to_num ensures that nan's are treated as 0's)
            if src.nodata is not None:
                resampled = np.where(resampled == src.nodata, np.nan, resampled)

            # Add to sum
            sum_array += np.nan_to_num(resampled)

# Update profile
profile.update(dtype=rasterio.float32, nodata=np.nan, compress="lzw")

# Write output
with rasterio.open(save_path, "w", **profile) as dst:
    dst.write(sum_array, 1)